# Módulo 3 · Clase 2: Clasificación — Predecir una Etiqueta, Decidir con Costo

**Machine Learning for Petroleum Engineers Using Python**  
SLB Ecuador · UDLA · 2026  

Instructor: **Carlos Enrique Mosquera Trujillo**  
Repo: [github.com/cmosquerat/slb-diplomado](https://github.com/cmosquerat/slb-diplomado)

---
## El mapa: dónde encaja lo de hoy

| | **Supervisado** | **No supervisado** |
|---|---|---|
| ¿Hay respuesta en los datos? | Sí, cada ejemplo trae su etiqueta o número | No, solo mediciones |
| ¿Qué hace? | **Predice** un valor nuevo | **Agrupa** lo que se parece |
| Ejemplo | "¿esta roca es reservorio?" | "agrupa estas profundidades" |

Dentro de **supervisado**, la pregunta que lo decide todo: **¿qué tipo de respuesta busco?**

- **Regresión** → *¿cuánto?* La respuesta es un **número** (el sónico es 84.2 µs/ft). ← clase anterior
- **Clasificación** → *¿cuál?* La respuesta es una **etiqueta** (esta roca es arenisca). ← **hoy**

## El problema de hoy

Tenemos los registros de un pozo nuevo y hay que decidir **qué intervalos completar y cañonear**. La **arenisca** almacena y deja fluir hidrocarburos (roca reservorio); la **lutita** no deja fluir nada (roca sello).

Los dos errores posibles **no cuestan lo mismo**:

- **Cañonear una zona seca** → gastamos en una completación que no produce.
- **Saltarnos una zona productiva** → dejamos petróleo en el subsuelo, y *nadie se entera*.

> 💡 Ejecuta con `Shift + Enter`. Las celdas 🧩 están **en blanco**: te toca escribirlas.

---
# 0 · Preparación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/cmosquerat/slb-diplomado/main/datos/litologia_force2020.csv"

lito = pd.read_csv(URL)
print(lito.shape)

---
# 1 · Conocer el dataset a fondo

**FORCE 2020**: dataset abierto de una competencia noruega de ML, con registros de 11 pozos del Mar del Norte (el mismo vecindario de Volve). Lo importante: **la litología la interpretó un geólogo**, no una fórmula.

## Las variables, una por una

Cada columna es un **sensor distinto bajado al pozo**. Entenderlas es entender el problema:

| Columna | Unidad | Qué mide físicamente | Por qué ayuda a distinguir la roca |
|---------|--------|----------------------|-------------------------------------|
| `GR` | GAPI | Radioactividad natural de la roca | Las **arcillas** concentran elementos radiactivos (potasio, torio); las **arenas limpias** casi no. GR alto → lutita |
| `RHOB` | g/cm³ | Densidad total (la herramienta emite rayos gamma y mide cuántos rebotan) | Una arena **porosa** tiene huecos llenos de fluido → pesa menos que una roca compacta |
| `NPHI` | fracción | Respuesta al bombardeo de neutrones: mide **hidrógeno** | El hidrógeno está en el agua y el petróleo de los poros… pero la arcilla **retiene agua ligada** y finge porosidad. NPHI alto suele ser lutita |
| `DTC` | µs/ft | Sónico: cuánto tarda el sonido en cruzar 1 pie (la variable que predijimos la clase pasada) | La roca compacta transmite **rápido** (DTC bajo); la roca blanda o porosa, lento |
| `RDEP` | ohm·m | Resistividad profunda: resistencia al paso de corriente | El agua salada **conduce**; el petróleo **no**. Habla del fluido, no tanto de la roca |
| `PROF` | m | Profundidad de la medición | Identifica la fila |
| `pozo` | — | Nombre del pozo | 11 pozos distintos |
| **`LITH`** | — | **La etiqueta del geólogo** | `Sandstone` o `Shale` |

> ⚠️ Fíjate: **`LITH` es texto**. El modelo no puede multiplicar la palabra *Shale* por un coeficiente — tendremos que convertirla a número. Eso lo haremos en la sección 2 (**binarización**).

In [ ]:
lito.head()

In [ ]:
lito.info()

## Estadísticas: el retrato numérico

`describe()` sobre las 5 mediciones. Leámoslo con ojos de petrofísico:

In [ ]:
feats = ["GR", "RHOB", "NPHI", "DTC", "RDEP"]
lito[feats].describe().round(2)

**Lectura física:** el `GR` va de casi 0 a ~195 API (todo el rango de arena limpia a arcilla pura); `RHOB` entre ~1.5 y ~3.0 g/cm³ (rango sedimentario normal); `RDEP` tiene una cola muy larga — hay valores extremos que habrá que tener en cuenta.

## ¿De qué pozos vienen los datos?

In [ ]:
lito["pozo"].value_counts()

> ⚠️ **Detalle importante para después:** los datos vienen de **pozos distintos**. Lo más honesto sería entrenar con unos pozos y evaluar con **otros** (un pozo nuevo de verdad). Hoy haremos un split aleatorio por simplicidad, pero tenlo presente: es una decisión metodológica real.

## El balance de clases: la alerta temprana

In [ ]:
lito["LITH"].value_counts()

In [ ]:
lito["LITH"].value_counts(normalize=True).round(3)

**83 % lutita, 17 % arenisca.** En geología es lo normal: la mayor parte de una columna sedimentaria es arcilla.

🚨 **Guarda este número.** Un "modelo" que responda **siempre lutita**, sin mirar ni un dato, acertaría el **83 %** de las veces. Lo retomaremos cuando hablemos de métricas.

## ¿Las mediciones distinguen una roca de la otra?

Comparemos el promedio de cada variable en cada clase:

In [ ]:
lito.groupby("LITH")[feats].mean().round(2)

El `GR` medio de la lutita (85) es **mucho mayor** que el de la arenisca (53) — justo lo que dice la geología. El `DTC` también separa bien (130 vs 103).

### Verlo: histogramas por clase

In [ ]:
arena = lito[lito["LITH"] == "Sandstone"]
lutita = lito[lito["LITH"] == "Shale"]

ax = lutita["GR"].plot(kind="hist", bins=60, alpha=0.6, label="lutita")
arena["GR"].plot(kind="hist", bins=60, alpha=0.6, ax=ax, label="arenisca")
ax.set_xlim(0, 200); ax.set_xlabel("GR (API)"); ax.legend()
ax.set_title("GR: separa... pero se traslapa")

**La lección clave del gráfico:** las clases se separan, **pero se traslapan en el medio**. No existe un corte de GR que las divida perfecto. Por eso el modelo no dará certezas: dará **probabilidades**.

### Dos variables a la vez

In [ ]:
ax = lutita.plot(kind="scatter", x="GR", y="RHOB", s=2, alpha=0.2,
                 color="gray", label="lutita")
arena.plot(kind="scatter", x="GR", y="RHOB", s=2, alpha=0.3,
           color="orange", ax=ax, label="arenisca")
ax.set_title("¿Se separan las dos clases?")

### ¿Qué tan correlacionada está cada variable con la etiqueta?

In [ ]:
# para correlacionar necesitamos un numero: version rapida (la formalizamos luego)
tmp = lito.copy()
tmp["es_arenisca"] = (tmp["LITH"] == "Sandstone").astype(int)

tmp[feats + ["es_arenisca"]].corr()["es_arenisca"].round(2).sort_values()

### La matriz de correlación completa

No solo contra el target: también **entre las variables**. Ahí aparece la **colinealidad** (información repetida), que vuelve inestables los coeficientes.

In [ ]:
import seaborn as sns

corr = tmp[feats + ["es_arenisca"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)

`NPHI` y `DTC` están muy correlacionadas **entre ellas**: las dos hablan de porosidad. Es información repetida — no arruina el modelo, pero conviene saberlo.

### Boxplots: las cinco variables de un vistazo

In [ ]:
lito.boxplot(column=feats, by="LITH", figsize=(12, 3.5), layout=(1, 5))

### El crossplot neutrón–densidad (el clásico petrofísico)

El gráfico que un petrofísico mira todos los días. Ojo: el eje de densidad va **invertido** por convención.

In [ ]:
ax = lutita.plot(kind="scatter", x="NPHI", y="RHOB", s=3, alpha=0.2,
                 color="gray", label="lutita", figsize=(6, 4.5))
arena.plot(kind="scatter", x="NPHI", y="RHOB", s=3, alpha=0.3,
           color="orange", ax=ax, label="arenisca")
ax.invert_yaxis()
ax.set_title("Crossplot neutron-densidad")

La **arenisca** se agrupa a la izquierda (poca respuesta de neutrón); la **lutita** se corre a la derecha porque la arcilla retiene agua ligada y *finge* porosidad.

### ¿Cómo se ve esto en un pozo real?

Los mismos datos en el formato que lee un geólogo: contra la **profundidad**.

In [ ]:
w = "15/9-13"
p = tmp[(tmp["pozo"] == w) & (tmp["PROF"] > 2000) & (tmp["PROF"] < 2400)]
p = p.sort_values("PROF")

fig, axes = plt.subplots(1, 4, figsize=(9, 6), sharey=True)
axes[0].plot(p["GR"], p["PROF"], lw=0.6, color="green");  axes[0].set_xlabel("GR")
axes[1].plot(p["RHOB"], p["PROF"], lw=0.6, color="red");   axes[1].set_xlabel("RHOB")
axes[2].plot(p["DTC"], p["PROF"], lw=0.6, color="blue");   axes[2].set_xlabel("DTC")
axes[3].fill_betweenx(p["PROF"], 0, p["es_arenisca"], color="orange")
axes[3].set_xlabel("reservorio")
axes[0].invert_yaxis(); axes[0].set_ylabel("Profundidad (m)")
fig.suptitle(f"Pozo {w}")

Donde el `GR` **baja**, aparece una barra naranja. **Eso** es lo que el modelo debe aprender: reproducir la cuarta pista a partir de las tres primeras.

### El balance de clases **por pozo**

In [ ]:
balance = tmp.groupby("pozo")["es_arenisca"].mean().sort_values() * 100
balance.round(1)

In [ ]:
balance.plot(kind="barh", figsize=(7, 4),
             title="% de arenisca en cada pozo")

Hay pozos con **8 %** de arenisca y otros con **72 %**: son geologías distintas. Al partir train/test **al azar** mezclamos profundidades del mismo pozo en ambos lados, así que el modelo ya "conoce" ese pozo. Lo más honesto sería **dejar pozos completos por fuera** para probar.

---
## 🧩 Práctica 1: Explorar antes de modelar

1. Haz el histograma de `DTC` por clase (como el de `GR`). ¿Separa mejor o peor?
2. Scatter de `NPHI` vs `RHOB` coloreado por clase. ¿Se ven dos nubes?
3. ¿Cuál es el `GR` **mediano** de cada clase? (`groupby` + `median`)
4. Según la tabla de correlaciones, ¿qué variable parece la menos útil? ¿Coincide con lo que dice la física?

In [ ]:
# Escribe tu solucion aqui


---
# 2 · De la recta a la probabilidad

## ¿Por qué no usar la regresión lineal?

Podríamos poner `0` = lutita, `1` = arenisca y ajustar una recta. El problema: **la recta no se detiene** en 0 ni en 1 — predice `1.3` o `-0.2`, y *"1.3 de arenisca"* no significa nada.

## La solución: la sigmoide

Una función que toma **cualquier** número y lo aplasta entre 0 y 1, para leerlo como **probabilidad**.

In [ ]:
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 200)
sigmoide = 1 / (1 + np.exp(-z))

plt.plot(z, sigmoide)
plt.axhline(0.5, ls='--', color='gray')
plt.xlabel('salida de la recta'); plt.ylabel('probabilidad')
plt.title('La sigmoide')

## ¿Qué es la regresión logística?

Pese al nombre, **no** sirve para predecir números: es el modelo base para **clasificar**. Por dentro:

`registros → una recta (suma ponderada) → sigmoide → probabilidad → etiqueta (según el umbral)`

Se llama "regresión" porque **internamente** hace una regresión; lo que **entrega** es una clasificación.

**Por qué empezar por ella:** se puede **explicar** (cada coeficiente dice cómo influye cada registro), es rápida y estable, y sirve de **línea base**: si un modelo complejo no la supera, no vale la pena su complejidad. **Su límite:** separa las clases con una frontera **recta**.

## Preprocesamiento: binarizar el target

El modelo **no entiende texto**: no puede multiplicar la palabra *Shale* por un coeficiente. **Binarizar** = convertir las dos categorías en `0` y `1`. Es parte del **preprocesamiento**: todo lo que hay que hacerle al dato *antes* de que el modelo lo pueda usar.

⚠️ **La decisión que define todo:** elegimos **arenisca = 1** porque es lo que **queremos detectar** (la *clase positiva*). Eso fija el significado de todo lo que viene: el **recall** será *"% de areniscas encontradas"*. Si pusiéramos lutita = 1, las mismas métricas medirían **otra cosa**.

> **Regla:** la clase positiva (`1`) es siempre **el evento raro que queremos cazar** — la zona productiva, la falla del equipo, la fuga de gas.

*(Si hubiera 3 o más categorías se usaría **one-hot encoding**: `pd.get_dummies`, una columna 0/1 por categoría.)*

In [ ]:
lito["es_reservorio"] = (lito["LITH"] == "Sandstone").astype(int)

lito[["LITH", "es_reservorio"]].head()

In [ ]:
# verificar que la conversion quedo bien: arenisca->1, lutita->0
lito.groupby("LITH")["es_reservorio"].mean()

## Entrenar: el mismo patrón de sklearn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = lito[feats]
y = lito["es_reservorio"]   # 1 = arenisca

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# estandarizar: ajustar SOLO con train (evitar data leakage)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_tr_s, y_tr)

> `stratify=y` mantiene la misma proporción de clases en train y test. Con datos desbalanceados, **siempre**.

## Leer el modelo: ¿en qué se fijó?

In [ ]:
pd.Series(modelo.coef_[0], index=feats).round(2).sort_values()

Coeficiente **negativo** = ese registro alto empuja hacia **lutita**. `GR` negativo: mucha radioactividad → arcilla. **Coincide con la geología** — buena señal de que el modelo no está memorizando ruido.

## Lo que realmente devuelve: probabilidades

In [ ]:
proba = modelo.predict_proba(X_te_s)[:, 1]
proba[:6].round(2)

In [ ]:
# para dar una respuesta hay que fijar un umbral (por defecto 0.5)
pred = (proba >= 0.5).astype(int)
pred[:6]

---
# 3 · Las métricas: aquí se decide si el modelo sirve

## La analogía del detector de gas

Ya saben evaluar un clasificador: conviven con uno.

- Un detector que **nunca suena** acierta el 99.9 % de los días (casi nunca hay fuga). **Accuracy altísima… y es un adorno.**
- Un detector que **suena siempre** no se pierde ninguna fuga, **pero nadie le cree**.

Nadie evalúa un detector por su % de aciertos: lo evalúa por **cuántas fugas detecta** (recall) y **cuántas veces molesta en falso** (precision).

## Accuracy: la métrica que engaña

In [ ]:
from sklearn.metrics import accuracy_score

print('accuracy:', round(accuracy_score(y_te, pred), 3))
print('si dijera SIEMPRE lutita:', round(1 - y_te.mean(), 3))

94.4 % suena excelente… pero el modelo tonto saca **82.8 %**. Solo mejoramos **12 puntos** sobre no pensar. Con clases desbalanceadas la accuracy **esconde** lo que importa: ¿encontramos las zonas de reservorio, que son las escasas?

## Abrir la caja: la matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_te, pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print()
print('correctos sello     (TN):', tn)
print('CAÑONEO EN SECO     (FP):', fp)
print('ZONA PERDIDA        (FN):', fn)
print('correctos reservorio(TP):', tp)

**Los cuatro resultados, con nombre de negocio:**

| | predije sello | predije reservorio |
|---|---|---|
| **ES sello** | correcto (12 680) | **falso positivo** → cañoneo en seco (323) |
| **ES reservorio** | **falso negativo** → zona perdida (563) | correcto (2 132) |

La accuracy sumaba todo esto en un número y **escondía los 563**.

## Precision y recall: dos preguntas distintas

Ambas salen de la misma matriz; la diferencia es **qué se toma como referencia**:

- **RECALL** — *"de las que **eran** reservorio, ¿cuántas encontré?"* → mira la **fila** de la realidad: `2132/(2132+563)`
- **PRECISION** — *"de las que **dije** reservorio, ¿cuántas acerté?"* → mira la **columna** de mis anuncios: `2132/(2132+323)`

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print('precision:', round(precision_score(y_te, pred), 3))
print('recall   :', round(recall_score(y_te, pred), 3))
print('f1       :', round(f1_score(y_te, pred), 3))

**Recall 0.79 significa: de cada 10 zonas de reservorio, se nos escapan 2.**

### ¿Cuál priorizar? Depende del daño

| Situación | Priorizar | Por qué |
|-----------|-----------|---------|
| Detectar zonas de reservorio | **Recall** | perderse una zona cuesta reservas para siempre |
| Alarma de fuga de gas | **Recall** | no detectarla es un evento de seguridad |
| Recomendar un workover de \$2M | **Precision** | equivocarse gasta el presupuesto en vano |
| Parar producción por alerta | **Precision** | cada parada en falso cuesta producción diferida |

⚠️ Subir una **casi siempre** baja la otra. La pregunta correcta no es *¿cuál es mejor?* sino **¿cuál error hace más daño aquí?**

### Criterios y valores aceptables

| Métrica | Cuándo elegirla | Qué es "aceptable" |
|---------|-----------------|---------------------|
| Accuracy | solo si las clases están **balanceadas** | debe superar **claramente** el % de la clase mayoritaria |
| Precision | cuando una **falsa alarma** cuesta cara | > 0.8 en decisiones operativas |
| Recall | cuando **perderse un caso** cuesta caro | > 0.9 si el caso perdido es crítico |
| F1 | comparar modelos con clases desbalanceadas | > 0.7 razonable · > 0.85 muy bueno |

**Nunca reportar una sola métrica.** Precision y recall siempre van juntas, y todas sobre el **test**.

---
## 🧩 Práctica 2: Interpretar el modelo

1. ¿Cuántas zonas de reservorio hay **en total** en el test? (`y_te.sum()`)
2. De esas, ¿cuántas encontró el modelo? Verifica que `recall = TP / (TP + FN)` a mano.
3. Calcula la **precision a mano** con `tp` y `fp`, y compárala con `precision_score`.
4. Si este modelo se usara para decidir cañoneos, ¿qué le preocuparía más al gerente de yacimientos: los 323 o los 563? ¿Por qué?

In [ ]:
# Escribe tu solucion aqui


---
# 4 · El costo de negocio: lo que separa a un ingeniero

Hasta ahora **contamos** errores. Ahora vamos a **ponerles precio**.

- **Falso positivo** (cañoneo en seco): gastamos una completación. Es dinero **medible** y se detecta rápido.
- **Falso negativo** (zona perdida): perdemos las reservas que esa zona habría producido **toda la vida del pozo**. Y **nadie se entera**.

Supongamos —conversándolo con el equipo de yacimientos— que **perder una zona cuesta 10 veces** un cañoneo en seco.

## El umbral: la perilla que casi nadie toca

Bajar el umbral = ser **más generoso** al declarar reservorio: encontramos más… y nos equivocamos más.

In [ ]:
COSTO_FN = 10   # perder una zona productiva
COSTO_FP = 1    # cañonear en seco

filas = []
for umbral in [0.5, 0.4, 0.3, 0.25, 0.2, 0.13, 0.1]:
    p = (proba >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, p).ravel()
    filas.append({
        "umbral": umbral,
        "recall": round(recall_score(y_te, p), 3),
        "precision": round(precision_score(y_te, p), 3),
        "zonas_perdidas_FN": fn,
        "canoneos_secos_FP": fp,
        "costo_total": fn * COSTO_FN + fp * COSTO_FP,
    })

tabla = pd.DataFrame(filas)
tabla

### La curva del costo

In [ ]:
umbrales = np.linspace(0.05, 0.95, 60)
costos = []
for u in umbrales:
    p = (proba >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, p).ravel()
    costos.append(fn * COSTO_FN + fp * COSTO_FP)

costos = np.array(costos)
mejor = umbrales[costos.argmin()]

plt.plot(umbrales, costos)
plt.axvline(0.5, ls='--', color='gray', label='umbral por defecto')
plt.axvline(mejor, color='green', label=f'minimo costo ({mejor:.2f})')
plt.xlabel('umbral'); plt.ylabel('costo total'); plt.legend()
plt.title('El umbral que minimiza el COSTO, no el error')

In [ ]:
print('umbral optimo:', round(mejor, 2))
print('costo en 0.5 :', int(costos[np.argmin(abs(umbrales - 0.5))]))
print('costo optimo :', int(costos.min()))

## El veredicto

| | Umbral 0.5 (por defecto) | Umbral 0.13 (por costo) |
|---|---|---|
| Accuracy | **0.944** | 0.895 |
| Recall | 0.791 | **0.905** |
| Zonas perdidas | 563 | **257** |
| **Costo total** | 5 953 | **3 962** |

El segundo modelo tiene **peor accuracy**. Un reporte automático diría que es inferior. **Y sin embargo es el que le conviene a la empresa** (33 % menos costo) — sin cambiar el modelo ni conseguir más datos: solo moviendo una perilla.

> ⭐ **Eso es lo que separa** al ingeniero que entiende el negocio del que solo corre `.fit()` y reporta la accuracy.

### Cómo se hace esto en la práctica

1. **Preguntar, no asumir**: sentarse con yacimientos o finanzas. *¿Cuánto cuesta cada error?* Casi nunca está escrito.
2. **Basta la proporción**: no hace falta el número exacto; con saber que un error cuesta ~10 veces el otro ya se puede decidir.
3. **Elegir el umbral con esa proporción**, no con el 0.5 de fábrica.
4. **Reportar en el lenguaje del que decide**: no *"recall 0.91"* sino *"recuperamos 306 zonas que antes se perdían, a cambio de 1 069 cañoneos adicionales"*.

---
## 🧩 Práctica integradora: tú eliges el umbral

El equipo de yacimientos revisó los números y te dice que en **este campo** perder una zona productiva cuesta **apenas 3 veces** un cañoneo en seco (las zonas son delgadas y hay muchos pozos vecinos).

1. Recalcula la tabla de costos con `COSTO_FN = 3`.
2. Grafica la curva de costo y encuentra el **nuevo umbral óptimo**.
3. ¿Subió o bajó respecto a 0.13? **¿Por qué tiene sentido?**
4. Con ese umbral: ¿cuántas zonas se pierden y cuántos cañoneos en seco se hacen?
5. Escribe **una frase** para el gerente explicando tu recomendación (sin usar la palabra *recall*).

In [ ]:
# Escribe tu solucion aqui


---
## Cierre

**El flujo de un problema de clasificación:**

`datos etiquetados` → `train/test` → `entrenar (.fit)` → `matriz de confusión` → **`umbral por costo`**

Los primeros 3 pasos son idénticos a la regresión de la clase pasada. **Los últimos 2 son los que hacen la diferencia** — y los que casi nadie hace.

**Conceptos:** supervisado vs no supervisado · regresión vs clasificación · sigmoide, probabilidad y umbral · desbalance y por qué la accuracy engaña · matriz de confusión · precision, recall, F1 · **matriz de costos y umbral óptimo**.

**Herramientas:** `LogisticRegression` · `.predict_proba()` · `confusion_matrix` · `precision_score`, `recall_score`, `f1_score`.

> **La idea para llevarse:** un modelo no se evalúa con una métrica. Se evalúa con **la decisión que habilita** y **lo que cuesta equivocarse**.

---
Carlos Enrique Mosquera Trujillo · cmosquerat@unal.edu.co  
**Machine Learning for Petroleum Engineers Using Python** · SLB Ecuador · UDLA · 2026

*Datos: FORCE 2020 Machine Learning Contest (Mar del Norte, Noruega) — dataset abierto.*